# Task 1: Extract Head-to-Head (H2H) Data
This notebook extracts Head-to-Head match data from the API and saves the raw JSON response to the `data/raw/` directory.

In [7]:
import os
import json
import time
from datetime import date

import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_FOOTBALL_KEY")
BASE_URL = "https://v3.football.api-sports.io/fixtures/headtohead"
HEADERS = {"x-apisports-key": API_KEY}

RAW_DIR = "../data/raw"

In [8]:
def get_h2h(team1_id: int, team2_id: int, max_retries: int = 3) -> dict:
    """
    Fetch the full head-to-head history between two teams.
    Handles timeouts and rate limiting (429).
    No 'season' or 'last' params: free plan doesn't support 'last',
    and omitting 'season' returns the full history across all seasons.
    No 'page' param either: this endpoint returns all results in
    a single response, it doesn't support pagination.
    """
    params = {
        "h2h": f"{team1_id}-{team2_id}",
    }

    response = None
    for attempt in range(max_retries):
        try:
            response = requests.get(
                BASE_URL, headers=HEADERS, params=params, timeout=15
            )
        except requests.exceptions.Timeout:
            print(f"Timeout on attempt {attempt + 1}, retrying...")
            time.sleep(5)
            continue

        if response.status_code == 429:
            wait = 30 * (attempt + 1)
            print(f"Rate limited (429). Waiting {wait}s before retry...")
            time.sleep(wait)
            continue

        break

    if response is None:
        raise RuntimeError(
            f"Failed to reach API after {max_retries} attempts "
            f"(team1={team1_id}, team2={team2_id})"
        )

    response.raise_for_status()
    data = response.json()

    return {
        "get": "fixtures/headtohead",
        "parameters": {"h2h": f"{team1_id}-{team2_id}"},
        "errors": data.get("errors"),
        "results": data.get("results", 0),
        "response": data.get("response", []),
    }

In [9]:
def save_raw(data: dict, team1_id: int, team2_id: int) -> str:
    os.makedirs(RAW_DIR, exist_ok=True)
    filename = f"{RAW_DIR}/h2h_{team1_id}_{team2_id}_{date.today().isoformat()}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return filename

In [16]:
import itertools

PROGRESS_FILE = f"{RAW_DIR}/h2h_progress.json"

# Current 2026-27 season SPL roster (18 teams)
TEAM_IDS = [
    2928,   # Al Khaleej
    2929,   # Al-Ahli Jeddah
    2930,   # Al-Faisaly FC (promoted 2026)
    2931,   # Al-Fateh
    2932,   # Al-Hilal
    2933,   # Al-Qadisiyah
    2934,   # Al-Ettifaq
    2936,   # Al Taawon
    2938,   # Al-Ittihad
    2939,   # Al-Nassr
    2940,   # Al Shabab
    2944,   # Al-Fayha
    2945,   # Al-Hazm (promoted 2025)
    2951,   # Abha (promoted 2026)
    10509,  # Al Kholood
    10511,  # Al Riyadh
    10513,  # Neom (promoted 2025)
    26738,  # Al Diriyah (promoted 2026)
]

DAILY_LIMIT = 90  # stay a bit under the 100/day cap for safety


def load_progress() -> set:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def save_progress(done: set) -> None:
    os.makedirs(RAW_DIR, exist_ok=True)
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(done), f, indent=2)


all_pairs = list(itertools.combinations(sorted(TEAM_IDS), 2))
done = load_progress()

print(f"Total pairs needed: {len(all_pairs)}")
print(f"Already fetched (previous runs): {len(done)}")

calls_today = 0

for team1_id, team2_id in all_pairs:
    pair_key = f"{team1_id}-{team2_id}"

    if pair_key in done:
        continue

    if calls_today >= DAILY_LIMIT:
        print(f"\nReached today's safe limit ({DAILY_LIMIT} calls).")
        print("Progress saved. Re-run this cell tomorrow to continue.")
        break

    print(f"Fetching H2H: {team1_id} vs {team2_id}")
    data = get_h2h(team1_id, team2_id)
    print(f"  results: {data['results']}, errors: {data['errors']}")

    filename = save_raw(data, team1_id, team2_id)
    print(f"  saved to {filename}")

    done.add(pair_key)
    save_progress(done)
    calls_today += 1

    time.sleep(6)

print(f"\nDone for now. {len(done)}/{len(all_pairs)} pairs fetched total.")

Total pairs needed: 153
Already fetched (previous runs): 153

Done for now. 153/153 pairs fetched total.
